# Load Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from pyspark.sql.functions import round as sp_round
import builtins # Import builtin Python functions, used for rounding the age here

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import urllib.request
import os


# Download DataSets From Web 
NOTE: Uncomment the download section on first run.

Files are already present in /Volumes if previously downloaded.

In [0]:
# Install
# %pip install names-dataset

# urllib.request.urlretrieve(
#     'https://raw.githubusercontent.com/RHHlayhel/DataEngineering/main/Projects/Syro_Lebanese_Titanic/data/Titanic-Dataset.csv',
#     '/Volumes/workspace/default/titanic/Data/Titanic-Dataset.csv'
# )

# urllib.request.urlretrieve(
#     'https://raw.githubusercontent.com/RHHlayhel/DataEngineering/main/Projects/Syro_Lebanese_Titanic/data/AddendumData.csv',
#     '/Volumes/workspace/default/titanic/Data/AddendumData.csv'
# )

# # Fix the Addendum file
# with open('/Volumes/workspace/default/titanic/Data/AddendumData.csv', 'r', newline='') as infile, \
#      open('/Volumes/workspace/default/titanic/Data/AddendumData_fixed.csv', 'w', newline='') as outfile:

#     for i, line in enumerate(infile):
#         line = line.rstrip('\r\n')
#         if i == 0:
#             outfile.write(line + '\n')
#         else:
#             if line.startswith('"') and line.endswith('"'):
#                 line = line[1:-1]
#                 line = line.replace('""', '"')
#             outfile.write(line + '\n')

# # Delete the raw unfixed file
# os.remove('/Volumes/workspace/default/titanic/Data/AddendumData.csv')


print("Files ready!")
print(os.listdir('/Volumes/workspace/default/titanic/Data/'))

Files ready!
['Titanic-Dataset.csv', 'AddendumData_fixed.csv']


In [0]:
titanic_main_df = spark.read.format('csv').option('header','true').option('inferSchema','true').load("/Volumes/workspace/default/titanic/Data/Titanic-Dataset.csv")

titanic_addendum_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/titanic/Data/AddendumData_fixed.csv")



In [0]:
titanic_main_df.printSchema()
titanic_main_df.show(5)

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|   

In [0]:
titanic_addendum_df.printSchema()
titanic_addendum_df.show(5)

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch| Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+
|        892|       0|     3|    Kelly, Mr. James|  male|34.5|    0|    0| 330911| 7.8292| NULL|       Q|
|        893|       1|     3|Wilkes, Mrs. Jame...|female|47.0|    1|    0| 363272|    7.0| NULL|       S|
|      

## Create Enriched Titanic Dataframe



In [0]:
titanic_full = titanic_main_df.unionByName(titanic_addendum_df)


## Clean Titanic Data

In [0]:
titanic_full.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in titanic_full.columns
]).show() # Check for Nulls


titanic_full_cleaned = titanic_full.fillna({
    'Fare':     0.0,
    'Cabin':    'Unknown',
    'Embarked': 'Unknown'
})

duplicates = titanic_full_cleaned.groupBy(titanic_full_cleaned.columns) \
    .agg(count("*").alias("count")) \
    .filter(col("count") > 1)
duplicates.show() # No duplicates in the titanic data

# titanic_full_cleaned = titanic_full_cleaned.dropDuplicates()

+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+
|PassengerId|Survived|Pclass|Name|Sex|Age|SibSp|Parch|Ticket|Fare|Cabin|Embarked|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+
|          0|       0|     0|   0|  0|263|    0|    0|     0|   1| 1014|       2|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+

+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+-----+
|PassengerId|Survived|Pclass|Name|Sex|Age|SibSp|Parch|Ticket|Fare|Cabin|Embarked|count|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+-----+
+-----------+--------+------+----+---+---+-----+-----+------+----+-----+--------+-----+



## Clean Names Data

In [0]:
# Extract surnames
from names_dataset import NameDataset
nd = NameDataset()
n = 500000

# Get Lebanese and Syrian last names
lb_surnames = nd.get_top_names(n=n, country_alpha2='LB', use_first_names=False)['LB']
sy_surnames = nd.get_top_names(n=n, country_alpha2='SY', use_first_names=False)['SY']

# Combine and deduplicate
all_surnames = list(set(lb_surnames + sy_surnames))
print(f"Total unique Syro-Lebanese surnames: {len(all_surnames)}")

Total unique Syro-Lebanese surnames: 27165


## Transform to Spark DF

In [0]:
surnames_df = spark.createDataFrame(
    [Row(Surname=name) for name in all_surnames]
)

print(f"Total surnames loaded: {surnames_df.count()}")
surnames_df.show()


Total surnames loaded: 27165
+---------+
|  Surname|
+---------+
| الكناكري|
|      Elî|
|    Yeshi|
|     داري|
|   Yammin|
|   Ferzan|
|  Sarmine|
|   Kallas|
|   Dabour|
|   الرقوي|
|     بلبل|
|أبو زكريا|
|     زكار|
|    عابرة|
|  الشتيوي|
|Bou Karim|
|     رووح|
|Salah Din|
|     Rise|
|    حمادى|
+---------+
only showing top 20 rows


## Keep Only Latin Script Names


In [0]:
latin_only = "^[a-zA-Z\u00C0-\u024F\\s\\-\\']+$"

surnames_df_latin = surnames_df.filter(
    col("Surname").rlike(latin_only)
)

print(f"Total latin surnames: {surnames_df_latin.count()}")
surnames_df_latin.show()

Total latin surnames: 15639
+---------+
|  Surname|
+---------+
|      Elî|
|    Yeshi|
|   Yammin|
|   Ferzan|
|  Sarmine|
|   Kallas|
|   Dabour|
|Bou Karim|
|Salah Din|
|     Rise|
|   Mamari|
|Alabrahem|
|   Falaha|
|  Hojairy|
| Bou Diab|
|    Kibbi|
| Ahmad Sy|
|Ghazzaoui|
|   Albeak|
|   Hareri|
+---------+
only showing top 20 rows


## Check Nulls and Empty Entries

In [0]:
surnames_df_latin.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in surnames_df_latin.columns
]).show() # Check for Nulls

surnames_df_latin.select([
    count(when(
        col(c).isNull() | (trim(col(c)) == ""), c
    )).alias(c)
    for c in surnames_df_latin.columns
]).show() # Check for Empty Spaces

+-------+
|Surname|
+-------+
|      0|
+-------+

+-------+
|Surname|
+-------+
|      0|
+-------+



## Finally: Match the surnames in Titanic and in the names dataset

In [0]:
titanic_full_cleaned = titanic_full_cleaned \
    .withColumn('Surname',
        trim(split(col('Name'), ',').getItem(0))
    ) 

# Normalize to uppercase on both sides
titanic_normalized = titanic_full_cleaned \
    .withColumn('Surname_upper', upper(col('Surname')))

surname_normalized = surnames_df_latin \
    .withColumn('Surname_upper', upper(col('Surname')))

# Join against Titanic surnames
Syro_Leb_Passengers = titanic_normalized.join(
    surname_normalized.select('Surname_upper'),  # Select only the key column
    on='Surname_upper',
    how='inner'
).drop('Surname_upper').drop('Surname')

Syro_Leb_Passengers = Syro_Leb_Passengers.sort('PassengerId',asecending=False)

print(f"Syro-Lebanese passengers found: {Syro_Leb_Passengers.count()}")
Syro_Leb_Passengers.show(5)


Syro-Lebanese passengers found: 47
+-----------+--------+------+--------------------+------+----+-----+-----+------+-------+-------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|Ticket|   Fare|  Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+------+-------+-------+--------+
|         10|       1|     2|Nasser, Mrs. Nich...|female|14.0|    1|    0|237736|30.0708|Unknown|       C|
|         49|       0|     3| Samaan, Mr. Youssef|  male|NULL|    2|    0|  2662|21.6792|Unknown|       C|
|        115|       0|     3|Attalah, Miss. Ma...|female|17.0|    0|    0|  2627|14.4583|Unknown|       C|
|        123|       0|     2|Nasser, Mr. Nicholas|  male|32.5|    1|    0|237736|30.0708|Unknown|       C|
|        141|       0|     3|Boulos, Mrs. Jose...|female|NULL|    0|    2|  2678|15.2458|Unknown|       C|
+-----------+--------+------+--------------------+------+----+-----+-----+------+-------+-------+--------+
on

## Clean the Produced dataframe before analysis

In [0]:
mean_age = builtins.round(Syro_Leb_Passengers.select(mean('Age')).first()[0], 1)  # Avoid weird looking decimal ages

Syro_Leb_Passengers = Syro_Leb_Passengers.fillna({
    'Age':      mean_age,  # Replace with mean to not skew statistical analysis
    'Fare':     0.0,
    'Cabin':    'Unknown',
    'Embarked': 'Unknown'
})

# Data Analysis

In [0]:
Syro_Leb_Passengers.groupBy('Sex').count().orderBy('count',ascending=False).show()

+------+-----+
|   Sex|count|
+------+-----+
|  male|   35|
|female|   12|
+------+-----+



In [0]:
Syro_Leb_Passengers.select(
    sp_round(avg('Age'), 1).alias('Average_Age')
).show()

+-----------+
|Average_Age|
+-----------+
|       23.4|
+-----------+



In [0]:
Syro_Leb_Passengers.groupBy('Pclass').count().orderBy('Pclass').show(truncate=False)

+------+-----+
|Pclass|count|
+------+-----+
|1     |3    |
|2     |3    |
|3     |41   |
+------+-----+



In [0]:
Syro_Leb_Passengers.groupBy('Embarked').count().orderBy('count').show(truncate=False)

+--------+-----+
|Embarked|count|
+--------+-----+
|Q       |2    |
|S       |6    |
|C       |39   |
+--------+-----+



In [0]:
Syro_Leb_Passengers.groupBy('Survived','Sex').count().orderBy('count').show(truncate=False)

+--------+------+-----+
|Survived|Sex   |count|
+--------+------+-----+
|0       |female|3    |
|1       |male  |3    |
|1       |female|9    |
|0       |male  |32   |
+--------+------+-----+



In [0]:
# Survival rate as percentage
Syro_Leb_Passengers.groupBy('Survived') \
    .count() \
    .withColumn('Percentage',
        sp_round(col('count') / Syro_Leb_Passengers.count() * 100, 1)
    ).show()

# Age distribution by survival
Syro_Leb_Passengers.groupBy('Survived') \
    .agg(sp_round(avg('Age'), 1).alias('Avg_Age')) \
    .show()

# Fare analysis
Syro_Leb_Passengers.select(
    sp_round(avg('Fare'), 2).alias('Avg_Fare'),
    sp_round(min('Fare'), 2).alias('Min_Fare'),
    sp_round(max('Fare'), 2).alias('Max_Fare')
).show()

+--------+-----+----------+
|Survived|count|Percentage|
+--------+-----+----------+
|       1|   12|      25.5|
|       0|   35|      74.5|
+--------+-----+----------+

+--------+-------+
|Survived|Avg_Age|
+--------+-------+
|       1|   25.8|
|       0|   22.6|
+--------+-------+

+--------+--------+--------+
|Avg_Fare|Min_Fare|Max_Fare|
+--------+--------+--------+
|   23.53|    6.95|  512.33|
+--------+--------+--------+



# Data Visualization

In [0]:
df = Syro_Leb_Passengers.toPandas()
df['Survived_Label'] = df['Survived'].map({1: 'Survived', 0: 'Did Not Survive'})
df['Pclass_Label'] = df['Pclass'].map({1: '1st Class', 2: '2nd Class', 3: '3rd Class'})
df['Embarked_Label'] = df['Embarked'].map({'C': 'Cherbourg', 'S': 'Southampton', 'Q': 'Queenstown', 'Unknown': 'Unknown'})

In [0]:
# All plots in one dashboard
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Sex Distribution', 'Survival by Sex', 'Passenger Class',
        'Age Distribution', 'Embarkation Port', 'Fare by Class'
    ),
    specs=[
        [{'type': 'pie'}, {'type': 'bar'}, {'type': 'bar'}],
        [{'type': 'histogram'}, {'type': 'bar'}, {'type': 'box'}]
    ]
)

# 1. Sex Distribution
sex = df['Sex'].value_counts()
fig.add_trace(go.Pie(
    labels=sex.index, values=sex.values,
    marker_colors=['#3498db', '#e91e8c'],
    showlegend=False,
    hole=0.4  # Donut style
), row=1, col=1)

# 2. Survival by Sex — Grouped Bar
for survived, color in zip(['Survived', 'Did Not Survive'], ['#2ecc71', '#e74c3c']):
    subset = df[df['Survived_Label'] == survived]['Sex'].value_counts()
    fig.add_trace(go.Bar(
        name=survived, x=subset.index, y=subset.values,
        marker=dict(color=color),
        showlegend=True
    ), row=1, col=2)

# 3. Passenger Class — Bar
pclass = df['Pclass_Label'].value_counts().sort_index()
fig.add_trace(go.Bar(
    x=pclass.index, y=pclass.values,
    marker_color='steelblue', showlegend=False
), row=1, col=3)

# 4. Age Distribution — Histogram
fig.add_trace(go.Histogram(
    x=df['Age'], nbinsx=30,
    marker_color='steelblue', showlegend=False
), row=2, col=1)

fig.update_xaxes(
    dtick=5,
    ticks='outside',      # Show tick marks outside the plot
    ticklen=8,            # Length of tick marks
    tickwidth=2,          # Width of tick marks
    showticklabels=True,  # Ensure labels are visible
    row=2, col=1
)

# 5. Embarkation Port — Bar
embarked = df['Embarked_Label'].value_counts()
fig.add_trace(go.Bar(
    x=embarked.index, y=embarked.values,
    marker_color='mediumpurple', showlegend=False
), row=2, col=2)

# 6. Fare by Class — Box
for pclass, color in zip([1, 2, 3], ['#f1c40f', '#3498db', '#e67e22']):
    subset = df[df['Pclass'] == pclass]
    fig.add_trace(go.Box(
        y=subset['Fare'], name=f'{pclass} Class',
        marker_color=color, showlegend=False
    ), row=2, col=3)

fig.update_layout(
    title_text='Syro-Lebanese Titanic Passengers — Analysis Dashboard',
    title_font_size=20,
    height=700,
    barmode='group',
    template='plotly_white'
)

fig.show()